In [19]:
import yaml
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [20]:
# Configuration and paths
mac = 20
pheno_list_type = None
# pheno_list_type = 'biochemistry'
# pheno_list_type = 'overall_phenotype'

# Load phenotype configuration
pheno_config_path = "/home/dnanexus/ukbgym/phenotype_config.yaml"
with open(pheno_config_path) as f:
    pheno_config = yaml.safe_load(f)

if pheno_list_type is not None:
    pheno_list = pheno_config[pheno_list_type]

eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'

# Load annotation configuration
config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = pl.DataFrame(records).with_columns(
    pl.col("annotation_dir").cast(pl.Int8)
)

all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

category,annotation,color,label,annotation_dir
str,str,str,str,i8
"""plof""","""loftee_hc""","""#E31A1C""","""LOFTEE HC""",1
"""plof_consequences""","""consequence_frameshift_variant""","""#E31A1C""","""VEP Frameshift""",1
"""plof_consequences""","""consequence_stop_gained""","""#FB9A99""","""VEP Stop Gained""",1
"""plof_consequences""","""consequence_splice_donor_varia…","""#6A1B9A""","""VEP Splice Donor""",1
"""plof_consequences""","""consequence_splice_acceptor_va…","""#AB47BC""","""VEP Splice Acceptor""",1
…,…,…,…,…
"""vep_consequences""","""consequence_splice_acceptor_va…","""#AB47BC""","""VEP Splice Acceptor""",1
"""vep_consequences""","""consequence_start_lost""","""#E65100""","""VEP Start Lost""",1
"""vep_consequences""","""consequence_stop_lost""","""#FFB300""","""VEP Stop Lost""",1


In [21]:
# !dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/annotation_files/genebass352genes_olink371genes_annotated_251205.parquet -o /home/dnanexus/data_dir/genebass352genes_olink371genes_annotated_251205.parquet

# !dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/annotation_files/genebass352genes_olink371genes_annotated_251205_fillna.parquet -o /home/dnanexus/data_dir/genebass352genes_olink371genes_annotated_251205_fillna.parquet

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/annotations_fillna_ukbgym.parquet -o /home/dnanexus/data_dir/annotations_fillna_ukbgym.parquet

Error: path "/home/dnanexus/data_dir/annotations_fillna_ukbgym.parquet"
already exists but -f/--overwrite was not set


In [ ]:
anno = pl.scan_parquet("/home/dnanexus/data_dir/annotations_fillna_ukbgym.parquet")

min_range = -2000
max_range = +0

anno = (
    anno
    .filter(
        # Choose CDS
        (pl.col('vep_cds_relaxed')==True) &
        # ((pl.col('vep_cds_relaxed')==True) | (pl.col('mane_cds')==True)) &
        # (pl.col('non_mane_cds')==False) &

        # Choose non CDS only
        # ((pl.col('vep_cds_relaxed')==False) & (pl.col('mane_cds')==False) & (pl.col('non_mane_cds')==False)) &

        # Choose VEP consequence
        # (pl.col('consequence_missense_variant') == True) &
        # (pl.col('consequence_synonymous_variant') == True) &
        # (pl.col('consequence_5_prime_utr_variant') == True) &
        # (pl.col('consequence_upstream_gene_variant') == True) &
        # (pl.col('consequence_downstream_gene_variant') == True) &
        # (pl.col('consequence_intron_variant') == True) &

        # Choose MobiDB region
        # (pl.col('mobi_lip_full') == True) &
        # (pl.col('mobi_disorder_full') == True) &

        # Custom variant class filter
        # filter_expression &

        # Choose regulatory region
        # (pl.col('encode_eh_pr') == True) &
        # (pl.col('encode_all_tf') == True) &
        
        # Proximity to TSS
        # (pl.col('dist_to_tss') >= min_range) &
        # (pl.col('dist_to_tss') <= max_range) &

        # Only SNPs
        (pl.col('ref').str.len_chars()==1) & (pl.col('alt').str.len_chars()==1)
    )
)

existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]

anno = (
    anno
    .select(
        # set(['id', 'region', 'tss', 'strand', 'gene_length', 'gene_name', 'dist_to_tss']).union(set(existing_annos))
        set(['id', 'region']).union(set(existing_annos))
    )
    .collect(engine='streaming')
    # .drop_nulls()
)

anno

loftee_hc,pangolin_score,absplice_dna_max,absplice2_max,ted_domain,five_prime_utr_variant_consequence_ustop_lost,mobi_lip_full,promoterai,consequence_synonymous_variant,id,low_complexity_domain,consequence_frameshift_variant,loftee_disorder,region,am_pathogenicity,loftee_lc,five_prime_utr_variant_consequence_ustop_gained,five_prime_utr_variant_consequence_uaug_gained,loftee_lip,cadd_raw,score_pai3d,loftee_low_complexity,loftee_ted,consequence_stop_lost,mobi_curated_disorder_priority,delta_score,consequence_start_lost,consequence_splice_acceptor_variant,esmscoremissense,abexp_abs_max,consequence_splice_donor_variant,consequence_missense_variant,gpn_score,consequence_stop_gained,five_prime_utr_variant_consequence_uaug_lost,verphylop
i8,f32,f32,f32,bool,u8,bool,f32,i8,str,bool,i8,bool,str,f32,i8,u8,u8,bool,f32,f32,bool,bool,i8,bool,f32,i8,i8,f32,f32,i8,i8,f32,i8,u8,f32
0,0.03,0.003,0.000327,false,0,false,0.0,0,"""chr5:87376465:A:T""",false,0,false,"""ENSG00000145715""",0.2063,0,0,0,false,3.681756,0.699444,false,false,0,false,0.05,0,0,-4.246,0.060269,0,1,-11.54,0,0,8.687
1,0.06,0.003,0.002165,false,0,true,0.0,0,"""chr1:177933615:G:A""",false,0,false,"""ENSG00000120341""",0.0,0,0,0,true,7.69086,0.0,false,false,0,false,0.0,0,0,0.0,0.946601,0,0,-1.6,1,0,2.372
0,0.0,0.001,0.000033,false,0,false,-0.143,1,"""chr2:96816550:C:T""",false,0,false,"""ENSG00000168763""",0.0,0,0,0,false,0.528766,0.0,false,false,0,false,0.0,0,0,0.0,0.012463,0,0,0.36,0,0,-1.5
0,0.03,0.0,0.000034,false,0,true,0.0,0,"""chr2:178598858:G:A""",false,0,false,"""ENSG00000155657""",0.0,0,0,0,false,3.032575,0.0,false,false,0,false,0.08,0,0,-6.253,0.015385,0,1,-8.09,0,0,6.948
0,0.0,0.003,0.000321,true,0,false,0.0,0,"""chr2:128318064:C:T""",false,0,false,"""ENSG00000136720""",0.2002,0,0,0,false,4.428444,0.550982,false,false,0,false,0.0,0,0,-5.721,0.160049,0,1,-5.12,0,0,-0.102
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,0.03,0.003,0.002172,true,0,false,0.0032,0,"""chr4:15816634:C:A""",false,0,false,"""ENSG00000004468""",0.0,0,0,0,false,5.248261,0.0,false,true,0,false,0.01,0,0,0.0,1.39091,0,0,-6.74,1,0,-0.6
0,0.0,0.001,0.000033,false,0,false,-0.0234,1,"""chr15:78077599:C:A""",true,0,false,"""ENSG00000167202""",0.0,0,0,0,false,1.2186,0.0,false,false,0,false,0.0,0,0,0.0,0.013111,0,0,-4.14,0,0,0.709
0,0.0,0.001,0.000033,true,0,true,0.0,1,"""chr14:67783209:G:A""",false,0,false,"""ENSG00000072121""",0.0,0,0,0,false,0.213689,0.0,false,false,0,false,0.0,0,0,0.0,0.006189,0,0,-0.88,0,0,2.934


In [ ]:
selected_annos = anno_config_df.filter(
    (pl.col('category').is_in(['plof', 'missense', 'genetic_diversity', 'conservation', 'splicing', 'regulatory_nondir'])) # CDS
)['annotation'].to_list()

melted_anno = (
    anno

    .select(
        set(['id', 'region']).union(set(selected_annos))
    )

    .unpivot(
        index=["id", "region"],
        on=selected_annos,
        variable_name="annotation",
        value_name="annotation_score"
    )
    .with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )
)

melted_anno

id,region,annotation,annotation_score
str,str,str,f32
"""chr5:87376465:A:T""","""ENSG00000145715""","""loftee_hc""",0.0
"""chr1:177933615:G:A""","""ENSG00000120341""","""loftee_hc""",1.0
"""chr2:96816550:C:T""","""ENSG00000168763""","""loftee_hc""",0.0
"""chr2:178598858:G:A""","""ENSG00000155657""","""loftee_hc""",0.0
"""chr2:128318064:C:T""","""ENSG00000136720""","""loftee_hc""",0.0
…,…,…,…
"""chr4:15816634:C:A""","""ENSG00000004468""","""abexp_abs_max""",1.39091
"""chr15:78077599:C:A""","""ENSG00000167202""","""abexp_abs_max""",0.013111
"""chr14:67783209:G:A""","""ENSG00000072121""","""abexp_abs_max""",0.006189


In [24]:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var/loftee_mac20_quant_pheno_assocs_EURunrelated_appv_percentiles.parquet -o /home/dnanexus/data_dir/loftee_mac20_quant_pheno_assocs_EURunrelated_appv_percentiles.parquet

appv = pl.scan_parquet("/home/dnanexus/data_dir/loftee_mac20_quant_pheno_assocs_EURunrelated_appv_percentiles.parquet")

# Create a lazy frame with the unique keys
anno_keys = anno.select(pl.col('id').unique()).lazy()

# Chain the filter and the much faster semi join
appv = (
    appv
    .join(
        anno_keys, on='id', how='semi'
    )
    .filter(
        pl.col('n_individuals') <= mac
    )
    .select(
        ['id', 'phenotype', 'mean_pheno_value', 'n_individuals']
    )
)
# unique_phenotypes = appv.select('phenotype').unique().collect(engine='streaming').to_series()

Error: path "/home/dnanexus/data_dir/loftee_mac20_quant_pheno_assocs_EURunrela
ted_appv_percentiles.parquet" already exists but -f/--overwrite was not set


In [ ]:
# Get gene trait associations
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/regenie_127phenotypes_lofteeHC_mac20.parquet -o /home/dnanexus/data_dir/
# !dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/loftee_mac20_associations_bh_corrected.parquet -o /home/dnanexus/data_dir/

gene_trait_df = pl.read_parquet('/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20.parquet')

gene_trait_df = (
    gene_trait_df
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'effect'])
)
gene_trait_df

dxpy.utils.resolver.ResolutionError: Unable to resolve "regenie_127phenotypes_lofteeHC_mac20.parquet" to a data object or folder name in '/processed_data/REGENIE_results'


region,phenotype,effect,loftee_corr_dir
str,str,f64,f64
"""ENSG00000132855""","""apolipoprotein_a_int""",-0.46207,-1.0
"""ENSG00000052841""","""apolipoprotein_a_int""",-0.807306,-1.0
"""ENSG00000110243""","""apolipoprotein_a_int""",-0.449444,-1.0
"""ENSG00000118137""","""apolipoprotein_a_int""",-2.60164,-1.0
"""ENSG00000173064""","""apolipoprotein_a_int""",-0.873767,-1.0
…,…,…,…
"""ENSG00000182095""","""forced_expiratory_volume_in_1s…",-0.860502,-1.0
"""ENSG00000164741""","""forced_expiratory_volume_in_1s…",-0.642983,-1.0
"""ENSG00000205189""","""forced_expiratory_volume_in_1s…",-0.635227,-1.0


In [26]:
gene_trait_df['region'].n_unique(), gene_trait_df.shape[0]

(699, 2289)

In [27]:
# --- 1. Lazily prepare the filter keys ---
region_keys = gene_trait_df.lazy().select(pl.col('region').unique())
anno_ids_lazy = melted_anno.lazy().join(
    region_keys, on='region', how='semi'
).select(pl.col('id').unique())


# --- 2. Build the main lazy query plan step-by-step ---
# Start by filtering 'appv', then join everything together. Every operation returns another LazyFrame.
final_lazy_plan = (
    # A. Use a semi join for efficient filtering (more memory-safe than 'is_in')
    appv
    .join(anno_ids_lazy, on="id", how="semi")

    # B. Join the other two dataframes
    .join(
        melted_anno.lazy().drop([c for c in melted_anno.columns if 'is_nan' in c]),
        on="id", 
        how="inner"
    )
    .join(
        gene_trait_df.lazy(), 
        on=["region", "phenotype"], 
        how="inner"
    )

    # C. Apply the ranking (window function) lazily
    .with_columns(
        pl.col(c)
        .rank("average")
        .over(["region", "phenotype", "annotation"])
        .alias(f"{c}_rank")
        for c in ['annotation_score', 'mean_pheno_value']
    )

    # D. Apply the group_by and aggregation lazily
    # .group_by(["region", "gene_name", "phenotype", "annotation"])
    .group_by(["region", "phenotype", "annotation"])
    .agg(
        n_variants = pl.col("id").count(),
        correlation = pl.corr("annotation_score_rank", "mean_pheno_value_rank", propagate_nans=True)
    )
)

# --- 3. Execute the ENTIRE plan at once ---
# This is the ONLY time data is computed. The streaming engine handles the whole complex query in memory-safe chunks.
print("Executing the full lazy plan with the streaming engine...")
correlation_df = final_lazy_plan.collect(engine='streaming')
correlation_df

Executing the full lazy plan with the streaming engine...


region,phenotype,annotation,n_variants,correlation
str,str,str,u64,f64
"""ENSG00000079387""","""forced_vital_capacity_fvc_best…","""gpn_score""",382,0.040991
"""ENSG00000178177""","""arm_predicted_mass_right_int""","""absplice2_max""",1127,0.000826
"""ENSG00000129214""","""testosterone_int""","""absplice2_max""",537,-0.062272
"""ENSG00000153956""","""mean_time_to_correctly_identif…","""absplice2_max""",661,NaN
"""ENSG00000137834""","""haematocrit_percentage_int""","""abexp_abs_max""",543,-0.132049
…,…,…,…,…
"""ENSG00000101191""","""body_mass_index_bmi_impedance_…","""absplice_dna_max""",1829,-0.001094
"""ENSG00000173409""","""mean_corpuscular_haemoglobin_i…","""absplice2_max""",181,-0.021702
"""ENSG00000155657""","""pulse_rate_automated_reading_i…","""abexp_abs_max""",21562,0.022345


In [31]:
correlation_df.filter(pl.col('annotation')=='loftee_hc').write_parquet('/home/dnanexus/data_dir/regenie_127phenotypes_mac20_lofteeHC_correlations.parquet')

In [32]:
!dx upload /home/dnanexus/data_dir/regenie_127phenotypes_mac20_lofteeHC_correlations.parquet project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/regenie_127phenotypes_mac20_lofteeHC_correlations.parquet

[===========================================================>] Uploaded 32,332 of 32,332 bytes (100%) /home/dnanexus/data_dir/regenie_127phenotypes_mac20_lofteeHC_correlations.parquet
ID                                file-J5gf7Z0JzbZkv56bpkY05bXg
Class                             file
Project                           container-J5gbV20JzbZQfPyBv22kzf5p
Folder                            /
Name                              regenie_127phenotypes_mac20_lofteeHC_correlations.parquet
State                             closing
Visibility                        visible
Types                             -
Properties                        -
Tags                              -
Outgoing links                    -
Created                           Tue Jan 20 11:15:28 2026
Created by                        shubhankar
 via the job                      job-J5gbV18Jg0y08vPp7GP3XKq0
Last modified                     Tue Jan 20 11:15:29 2026
Media type                        
archivalState              

In [33]:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/regenie_127phenotypes_mac20_lofteeHC_correlations.parquet -o /home/dnanexus/data_dir/

loftee_corrs = (
    pl.read_parquet('/home/dnanexus/data_dir/regenie_127phenotypes_mac20_lofteeHC_correlations.parquet')
    .with_columns(
        loftee_corr = pl.col('correlation'),
        loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
    )
    .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_dir']) 
)

loftee_corrs

dxpy.utils.resolver.ResolutionError: Unable to resolve "regenie_127phenotypes_mac20_lofteeHC_correlations.parquet" to a data object or folder name in '/processed_data/REGENIE_results'


region,phenotype,loftee_corr,loftee_corr_dir
str,str,f64,f64
"""ENSG00000067900""","""lymphocyte_count_int""",-0.097882,-1.0
"""ENSG00000145386""","""mean_corpuscular_haemoglobin_i…",-0.059845,-1.0
"""ENSG00000164944""","""trunk_predicted_mass_int""",-0.04623,-1.0
"""ENSG00000107863""","""trunk_predicted_mass_int""",-0.122605,-1.0
"""ENSG00000121966""","""eosinophill_percentage_int""",-0.075765,-1.0
…,…,…,…
"""ENSG00000142208""","""arm_predicted_mass_right_int""",-0.049973,-1.0
"""ENSG00000101670""","""cholesterol_int""",0.035791,1.0
"""ENSG00000017427""","""weight_int""",-0.170599,-1.0
